# Checking if T4 GPU is used (before ssh)

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# Installing dependencies

In [3]:
# Install standard, stable versions compatible with the GPU 
%pip install -q ultralytics roboflow torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.7 MB/s eta 0:00:00


# Download Vietnamese License Plate Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="your_api_key")  # Replace with your actual Roboflow API key
# https://app.roboflow.com/alphajack2006-gmail-com/vietnamese-car-license-plate-vlbzz/train
# location of the dataset in Roboflow.
project = rf.workspace("alphajack2006-gmail-com").project("vietnamese-car-license-plate-vlbzz")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vietnamese-Car-License-Plate--1 in yolov8:: 100%|██████████| 16513/16513 [00:01<00:00, 8936.51it/s] 


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Applied Fine Tuning (leveraging a pretrained model to specialise it in finding license plates)

I did not use any optimisation paths like: adding shears or freezing any base layers.


# Train YOLOv8 Model

In [7]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    name="vietnam_plate_lpr"
)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vietnamese-Car-License-Plate--1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vietnam_plate_lpr-2, nbs=64, nms=None, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c9eed229b70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

also, on the side note: mp950-95 shows how tight the bounding boxes are fit around the license plate. since we're doing OCR within a bounding box, i'd need to manually expand the bounding boxes before pushing it to an OCR model that will extract the text.

# Download Trained best.pt Weights to Your Local Machine

In [10]:
from google.colab import files
best = YOLO("runs/detect/vietnam_plate_lpr/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr/weights/best.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model is downloaded to colab notebook's local runtime; please re-download best.pt from there and replace it in your directory's Models_yolo, if retrained.


If on VSCODE's colab extension .ipynb (what I used), go to the colab extension, click on the existing remote server, and find /runs/... to download the .pt model.